# Edge benchmark
Model size and prediction time on a **single CPU core** (a stand-in for a small edge device). Task: AD vs CN.

Accuracy for the main models comes from `model_comparison.csv`; the smaller random forest is evaluated here.

In [1]:
import sys
sys.path.insert(0, "..")

import pickle
import time
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from src.features import load_spectra, feature_table, bands_from_spectra, spectra as compute_spectra
from src.models import make_logreg, make_rf, CNNClassifier
from src.evaluate import repeat_cv
from src.config import RESULTS

torch.set_num_threads(1)                                  # one core only

def make_small_rf():
    # an edge-friendly forest: fewer, shallower trees
    return RandomForestClassifier(n_estimators=50, max_depth=8, class_weight="balanced", n_jobs=-1, random_state=42)

spec = load_spectra()
keep = np.isin(spec["label"], ["AD", "CN"])
power = spec["power"][keep]
y = (spec["label"][keep] == "AD").astype(int)
bands = bands_from_spectra(spec["freqs"], power)
flat = power.reshape(len(power), -1)

models = {"LogReg": (make_logreg().fit(bands, y), bands),
          "RandomForest": (make_rf().fit(bands, y), bands),
          "RandomForest (small)": (make_small_rf().fit(bands, y), bands),
          "CNN": (CNNClassifier().fit(flat, y), flat)}

In [2]:
def median_ms(fn, runs=200):
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return 1000 * np.median(times)

# feature step: one 4 s epoch of raw signal (19 channels x 2000 samples) -> spectra -> bands
epoch = np.random.default_rng(0).standard_normal((1, 19, 2000))
spectrum_ms = median_ms(lambda: compute_spectra(epoch, 500.0))
bands_ms = median_ms(lambda: bands_from_spectra(*compute_spectra(epoch, 500.0))) - spectrum_ms

# accuracy (subject-level CV) for each model
accuracy = pd.read_csv(RESULTS / "model_comparison.csv").query("task == 'AD-CN'").set_index("model").accuracy
small = repeat_cv(feature_table(spec), make_small_rf, ["AD", "CN"], n_seeds=10)
accuracy["RandomForest (small)"] = round(small["accuracy"], 3)

rows = []
for name, (model, X) in models.items():
    if hasattr(model, "n_jobs"):
        model.n_jobs = 1                                  # one core for prediction too
    one = X[:1]
    rows.append({"model": name,
                 "accuracy": accuracy[name],
                 "size_kb": round(len(pickle.dumps(model)) / 1024, 1),
                 "predict_ms_per_epoch": round(median_ms(lambda: model.predict_proba(one)), 3),
                 "features_ms_per_epoch": round(spectrum_ms + (0 if name == "CNN" else bands_ms), 3)})

bench = pd.DataFrame(rows)
bench["total_ms_per_epoch"] = bench.predict_ms_per_epoch + bench.features_ms_per_epoch
print(bench.to_string(index=False))
bench.to_csv(RESULTS / "edge_benchmark.csv", index=False)

               model  accuracy  size_kb  predict_ms_per_epoch  features_ms_per_epoch  total_ms_per_epoch
              LogReg     0.794      4.2                 0.286                  0.828               1.114
        RandomForest     0.822  58597.7                13.969                  0.828              14.797
RandomForest (small)     0.800   1544.9                 2.487                  0.828               3.315
                 CNN     0.677     54.3                 0.207                  0.789               0.996
